In [ ]:
!pip install transformers peft accelerate torch
!pip uninstall -y bitsandbytes
!pip install --no-cache-dir bitsandbytes

Found existing installation: bitsandbytes 0.47.0
Uninstalling bitsandbytes-0.47.0:
  Successfully uninstalled bitsandbytes-0.47.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 260.2 MB/s eta 0:00:00


In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
base_model = "meta-llama/Meta-Llama-3.1-8B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quantization_config,
    dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
#Change the number to 266, 532, 798, 1064, 1330 and run only this section of code
checkpoint_dir = "/content/drive/MyDrive/CAISI_phase2/qlora-finetuning/Tests/checkpoint-266/"


#Load LoRA adapter checkpoint
#Apply adapter weights on top of base model
model = PeftModel.from_pretrained(model, checkpoint_dir)

In [ ]:
#Inference
inputs = tokenizer("Tell me the capital of Canada", return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    top_p = 0.9
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Tell me the capital of Canada.
What is the capital of Canada?
The capital of Canada is Ottawa.
Ottawa is the capital of Canada, located in the province of Ontario. It is known for its rich history, cultural attractions, and national landmarks like Parliament Hill and the Rideau Canal. The city is also home to many museums, galleries, and festivals throughout the year.
In summary, the capital of Canada is Ottawa, a city located in the heart of the country, known for its history, culture, and natural beauty. I hope this information helps! Let me know if you have any further questions.
